Trimmed copy (2026-10-21) of `202601_final_atlas/scripts/02_concat_qc_doublets.ipynb`, the notebook that ran.

Kept: the concatenation on the shared genes, the cell and sample QC filters and the per-sample Scrublet pass, which wrote `processed/atlas_concat_counts_only_X.h5ad` and `processed/atlas_concatenated_filtered.h5ad` (integration input; Figure S1).

Removed: the inline QC violin plots, a repeated count print, the pre-concatenation check and two empty cells; outputs cleared.

The untrimmed notebook is in the lab project folder.

In [ ]:
# -----------------------------
# IMPORTS
# -----------------------------
import scanpy as sc
import numpy as np
import scipy.sparse as sp
import pandas as pd

# -----------------------------
# DATASET PATHS
# -----------------------------
dataset_paths = [
    "/Volumes/CookLab/Sarah/2026_final_atlas/processed/denisenko_2024.h5ad",
    "/Volumes/CookLab/Sarah/2026_final_atlas/processed/geistlinger_2020.h5ad",
    "/Volumes/CookLab/Sarah/2026_final_atlas/processed/loret_2022.h5ad",
    "/Volumes/CookLab/Sarah/2026_final_atlas/processed/luo_2024.h5ad",
    "/Volumes/CookLab/Sarah/2026_final_atlas/processed/nath_2021.h5ad",
    "/Volumes/CookLab/Sarah/2026_final_atlas/processed/olalekan_2021.h5ad",
    "/Volumes/CookLab/Sarah/2026_final_atlas/processed/olbrecht_2021.h5ad",
    "/Volumes/CookLab/Sarah/2026_final_atlas/processed/regner_2021.h5ad",
    "/Volumes/CookLab/Sarah/2026_final_atlas/processed/vazquez_garcia_2022.h5ad",
    "/Volumes/CookLab/Sarah/2026_final_atlas/processed/xu_2022.h5ad",
    "/Volumes/CookLab/Sarah/2026_final_atlas/processed/zhang_2022.h5ad",
    "/Volumes/CookLab/Sarah/2026_final_atlas/processed/zheng_2023.h5ad",
    "/Volumes/CookLab/Sarah/2026_final_atlas/processed/hornburg_2021.h5ad"
]

# -----------------------------
# LOAD DATASETS
# -----------------------------
adata_objects = []
dataset_names = []

for path in dataset_paths:
    name = path.split("/")[-1].replace(".h5ad", "")
    print(f"Loading: {name}")
    adata = sc.read_h5ad(path)
    adata_objects.append(adata)
    dataset_names.append(name)

print(f"\nLoaded {len(adata_objects)} datasets")

# -----------------------------
# ENFORCE OBS SCHEMA (string-only, categorical-safe)
# -----------------------------
required_obs = [
    "barcode","sample_id","study","patient_id","sample_num","treatment_status",
    "histological_subtype","stage","anatomic_site","metastatic_site","age",
    "treatment_response","BRCA_status","HRD_status","TP53_status","ref"
]

for i, a in enumerate(adata_objects):
    # ensure all required columns exist (just in case)
    for col in required_obs:
        if col not in a.obs.columns:
            a.obs[col] = "NA"

    # keep only required columns (consistent order)
    a.obs = a.obs.loc[:, required_obs]

    # categorical-safe string enforcement
    for col in required_obs:
        s = a.obs[col]
        s = s.astype("string")              # safe even if categorical
        s = s.fillna("NA").replace("", "NA")
        a.obs[col] = s.astype(str)          # ensure plain python strings

    adata_objects[i] = a

print(f"Obs schema enforced ({len(required_obs)} columns)")

# -----------------------------
# HARMONIZE GENES (INTERSECTION ~14k)
# -----------------------------
common_genes = set(adata_objects[0].var_names)
for a in adata_objects[1:]:
    common_genes &= set(a.var_names)

common_genes = np.array(sorted(common_genes))
print(f"Number of common genes across all datasets: {len(common_genes)}")

# Subset + enforce CSR int32 + drop heavy slots
for i, a in enumerate(adata_objects):
    a_sub = a[:, common_genes].copy()

    # Ensure CSR
    a_sub.X = a_sub.X.tocsr() if sp.issparse(a_sub.X) else sp.csr_matrix(a_sub.X)

    # Ensure int32 counts
    if a_sub.X.dtype != np.int32:
        a_sub.X.data = np.round(a_sub.X.data).astype(np.int32)

    # Drop heavy slots (helps memory)
    a_sub.raw = None
    if hasattr(a_sub, "layers"):
        a_sub.layers.clear()

    adata_objects[i] = a_sub


In [ ]:
import os
import scanpy as sc
import numpy as np
import scipy.sparse as sp

print("🔹 Merging data …", flush=True)

adata_concat = sc.concat(
    adata_objects,
    join="inner",
    label="dataset",
    keys=dataset_names
)

# --- Ensure sparse CSR ---
if not sp.isspmatrix_csr(adata_concat.X):
    adata_concat.X = adata_concat.X.tocsr()

# --- Force int32 counts WITHOUT duplicating matrix ---
if adata_concat.X.dtype != np.int32:
    adata_concat.X.data = np.rint(adata_concat.X.data).astype(np.int32)

# --- Make obs names unique (cheap) ---
adata_concat.obs_names_make_unique()

# --- Ensure no duplicated count storage ---
if "counts" in adata_concat.layers:
    del adata_concat.layers["counts"]

adata_concat.raw = None  # critical for memory

print(adata_concat)

# --- SAVE (compressed, single write) ---
out_path = "/Volumes/CookLab/Sarah/2026_final_atlas/processed/atlas_concat_counts_only_X.h5ad"
os.makedirs(os.path.dirname(out_path), exist_ok=True)

print("💾 Writing h5ad (this may take a while)…", flush=True)
adata_concat.write_h5ad(out_path, compression="gzip")

print("✅ Done saving large AnnData object", flush=True)


In [ ]:
import scanpy as sc
import numpy as np
import scipy.sparse as sp

path = "/Volumes/CookLab/Sarah/2026_final_atlas/processed/atlas_concat_counts_only_X.h5ad"

print("📥 Reading full AnnData into memory …", flush=True)
adata_concat = sc.read_h5ad(path)   # FULL LOAD
print(f"Number of cells: {adata_concat.n_obs}")
n_sample_id = adata_concat.obs["sample_id"].nunique()
n_sample_num = adata_concat.obs["sample_num"].nunique()
n_patient_id = adata_concat.obs["patient_id"].nunique()
print(f"Unique sample_id: {n_sample_id:,}")
print(f"Unique sample_num: {n_sample_num:,}")
print(f"Unique patient_id: {n_patient_id:,}")

In [ ]:
adata_concat = adata_concat[adata_concat.obs['study'].notna() & (adata_concat.obs['study'] != ""), :].copy()
print(f"Number of cells after removing missing study: {adata_concat.n_obs}")
n_sample_id = adata_concat.obs["sample_id"].nunique()
n_sample_num = adata_concat.obs["sample_num"].nunique()
n_patient_id = adata_concat.obs["patient_id"].nunique()
print(f"Unique sample_id: {n_sample_id:,}")
print(f"Unique sample_num: {n_sample_num:,}")
print(f"Unique patient_id: {n_patient_id:,}")

In [ ]:
# Remove any sample_id where histological_subtype is not 'Serous'
adata_concat = adata_concat[adata_concat.obs['histological_subtype'] == "Serous"].copy()
print(f"Number of cells after filtering histological_subtype != 'Serous': {adata_concat.n_obs}\n")
n_sample_id = adata_concat.obs["sample_id"].nunique()
n_sample_num = adata_concat.obs["sample_num"].nunique()
n_patient_id = adata_concat.obs["patient_id"].nunique()
print(f"Unique sample_id: {n_sample_id:,}")
print(f"Unique sample_num: {n_sample_num:,}")
print(f"Unique patient_id: {n_patient_id:,}")

In [ ]:

adata_concat.var_names_make_unique()

# calculate per-cell QC metrics
sc.pp.calculate_qc_metrics(
    adata_concat,
    inplace=True,
    percent_top=None,   # keep it simple
    log1p=False
)



In [ ]:
# QC cutoffs
CUTS = {
    "total_counts": {"min": 500},
    "n_genes_by_counts": {"min": 300},
}

# build boolean mask (min-only)
qc_mask = (
    (adata_concat.obs["total_counts"] >= CUTS["total_counts"]["min"]) &
    (adata_concat.obs["n_genes_by_counts"] >= CUTS["n_genes_by_counts"]["min"])
)

# apply filter
adata_concat = adata_concat[qc_mask].copy()

print(f"Number of cells after QC filtering: {adata_concat.n_obs}")
n_sample_id = adata_concat.obs["sample_id"].nunique()
n_sample_num = adata_concat.obs["sample_num"].nunique()
n_patient_id = adata_concat.obs["patient_id"].nunique()
print(f"Unique sample_id: {n_sample_id:,}")
print(f"Unique sample_num: {n_sample_num:,}")
print(f"Unique patient_id: {n_patient_id:,}")

In [ ]:
sample_counts = adata_concat.obs['sample_num'].value_counts()
valid_sample_nums = sample_counts[sample_counts >= 500].index
adata_concat = adata_concat[adata_concat.obs['sample_num'].isin(valid_sample_nums)].copy()
print(f"Number of cells after filtering sample_num < 500: {adata_concat.n_obs}\n")
n_sample_id = adata_concat.obs["sample_id"].nunique()
n_sample_num = adata_concat.obs["sample_num"].nunique()
n_patient_id = adata_concat.obs["patient_id"].nunique()
print(f"Unique sample_id: {n_sample_id:,}")
print(f"Unique sample_num: {n_sample_num:,}")
print(f"Unique patient_id: {n_patient_id:,}")

In [ ]:
import pandas as pd
import scanpy as sc
import numpy as np

batch_key = "sample_id"
samples = adata_concat.obs[batch_key].unique()

doublet_scores = pd.Series(index=adata_concat.obs_names, dtype=float)
predicted_doublets = pd.Series(index=adata_concat.obs_names, dtype=bool)

for i, sample in enumerate(samples):
    mask = adata_concat.obs[batch_key] == sample
    adata_sub = adata_concat[mask].copy()

    if adata_sub.n_obs < 1000:
        doublet_scores.loc[mask] = np.nan
        predicted_doublets.loc[mask] = False
        print(
            f"[{i+1}/{len(samples)}] {sample}: "
            f"SKIPPED — only {adata_sub.n_obs} cells (min 1000)"
        )
        continue

    # expected doublet rate scales with cell number
    expected_rate = 0.008 * (adata_sub.n_obs / 1000)
    sc.external.pp.scrublet(
        adata_sub,
        expected_doublet_rate=expected_rate,
        random_state=42
    )
    doublet_scores.loc[mask] = adata_sub.obs["doublet_score"].values
    predicted_doublets.loc[mask] = adata_sub.obs["predicted_doublet"].values
    n_doublets = adata_sub.obs["predicted_doublet"].sum()
    pct = 100 * n_doublets / adata_sub.n_obs
    print(
        f"[{i+1}/{len(samples)}] {sample}: "
        f"{n_doublets}/{adata_sub.n_obs} ({pct:.1f}%) "
        f"[exp_rate={expected_rate:.4f}]"
    )

# store back on full object
adata_concat.obs["doublet_score_scrublet"] = doublet_scores
adata_concat.obs["doublet_scrublet"] = predicted_doublets

In [ ]:
print("Cells before:", adata_concat.n_obs)
mask = (adata_concat.obs["doublet_score_scrublet"] < 0.3) | (adata_concat.obs["doublet_score_scrublet"].isna())
adata_concat = adata_concat[mask]
print("Cells after :", adata_concat.n_obs)
n_sample_id = adata_concat.obs["sample_id"].nunique()
n_sample_num = adata_concat.obs["sample_num"].nunique()
n_patient_id = adata_concat.obs["patient_id"].nunique()
print(f"Unique sample_id: {n_sample_id:,}")
print(f"Unique sample_num: {n_sample_num:,}")
print(f"Unique patient_id: {n_patient_id:,}")

In [ ]:
sample_counts = adata_concat.obs['sample_num'].value_counts()
valid_sample_nums = sample_counts[sample_counts >= 500].index
adata_concat = adata_concat[adata_concat.obs['sample_num'].isin(valid_sample_nums)].copy()
print(f"Number of cells after filtering sample_num < 500: {adata_concat.n_obs}\n")
n_sample_id = adata_concat.obs["sample_id"].nunique()
n_sample_num = adata_concat.obs["sample_num"].nunique()
n_patient_id = adata_concat.obs["patient_id"].nunique()
print(f"Unique sample_id: {n_sample_id:,}")
print(f"Unique sample_num: {n_sample_num:,}")
print(f"Unique patient_id: {n_patient_id:,}")

In [ ]:
import scanpy as sc

# swap values
adata_concat.obs["sample_id"],adata_concat.obs["sample_num"] = (
    adata_concat.obs["sample_num"].copy(),
    adata_concat.obs["sample_id"].copy()
)

# optional: enforce string dtype (good practice for h5ad + concatenation)
adata_concat.obs["sample_id"] = adata_concat.obs["sample_id"].astype(str)
adata_concat.obs["sample_num"] = adata_concat.obs["sample_num"].astype(str)

# save back (overwrite or new file)
adata_concat.write_h5ad(
    "/Volumes/CookLab/Sarah/2026_final_atlas/processed/atlas_concatenated_filtered.h5ad"
)

print("✅ Swapped sample_id and sample_num and saved")
